# 00 Baseline Eval

Sprint 1 baseline evaluation notebook.

This notebook intentionally keeps orchestration in the notebook and evaluation logic in `src/`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / "src").exists():
            ROOT = candidate
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT


In [ ]:
from src import load_config
from src.data import load_eval_examples, resolve_path
from src.eval import HeuristicPredictor, evaluate_examples
from src.eval.reporting import export_handoff_bundle
from src.eval.splits import split_examples
from src.prompts import get_prompt_variants
from src.solvers import ConservativeRouter

config = load_config(ROOT / "configs/experiments/baseline_eval.yaml")
all_examples = load_eval_examples(resolve_path(config["data"]["path"]))
train_examples, validation_examples = split_examples(
    all_examples,
    validation_ratio=config["split"]["validation_ratio"],
    seed=config["split"]["seed"],
)
selected_examples = validation_examples or all_examples
len(train_examples), len(selected_examples)


In [ ]:
selected_prompt_names = set(config["prompts"])
prompt_variants = [
    variant
    for variant in get_prompt_variants()
    if variant.name in selected_prompt_names
]
router = ConservativeRouter(
    confidence_threshold=config["routing"]["confidence_threshold"]
)
result = evaluate_examples(
    selected_examples,
    prompt_variants=prompt_variants,
    predictor=HeuristicPredictor(),
    router=router if config["routing"]["enabled"] else None,
    run_name=config["run_name"],
    output_dir=ROOT / config["reporting"]["output_dir"],
)
handoff_bundle = export_handoff_bundle(
    result,
    ROOT / config["kaggle"]["output_dir"],
    selected_variant=config["selected_prompt_variant"],
)
result.metrics


In [ ]:
summary_path = result.artifact_paths["summary_md"]
print(summary_path.read_text(encoding="utf-8"))
print(f"Handoff bundle: {handoff_bundle}")
